# PicoSAM3 — Teacher Precomputation
Runs SAM3 over COCO images and saves teacher logits to Google Drive.
Safe to interrupt and resume — already-cached annotations are skipped automatically.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import userdata
import os, wandb
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
from google.colab import userdata
wandb.login(key=userdata.get('WANDB_API_KEY'))


Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: orjanhh (orjanhh-eth-z-rich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2. Clone repo and install dependencies

In [ ]:
import os

REPO_DIR = '/content/picosam3'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/pbonazzi/picosam3 {REPO_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

Cloning into '/content/picosam3'...
remote: Enumerating objects: 1770, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 1770 (delta 11), reused 15 (delta 6), pack-reused 1744 (from 3)
Receiving objects: 100% (1770/1770), 587.35 MiB | 35.17 MiB/s, done.
Resolving deltas: 100% (184/184), done.
/content/picosam3


In [ ]:
!pip install -q ftfy regex pycocotools wandb huggingface_hub iopath decord
!pip install -q -e .

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 149.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.0 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done


## 3. Download COCO dataset to Google Drive
Only runs if files are not already present. Images are ~1GB (val2017) — use val2017 for development, switch to train2017 (~18GB) for full training.

**Change `USE_VAL` to `False` to use the full train2017 set.**

In [ ]:
USE_VAL = False  # Set to False for full train2017 (~18GB)

DRIVE_DATA_DIR = '/content/drive/MyDrive/picosam3/dataset'
LOCAL_DATA_DIR = '/content/dataset'
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

if USE_VAL:
    IMAGE_DIR = f'{LOCAL_DATA_DIR}/val2017'
    ANN_FILE  = f'{LOCAL_DATA_DIR}/annotations/instances_val2017.json'
    IMG_ZIP_URL = 'http://images.cocodataset.org/zips/val2017.zip'
    IMG_ZIP     = f'{DRIVE_DATA_DIR}/val2017.zip'
else:
    IMAGE_DIR = f'{LOCAL_DATA_DIR}/train2017'
    ANN_FILE  = f'{LOCAL_DATA_DIR}/annotations/instances_train2017.json'
    IMG_ZIP_URL = 'http://images.cocodataset.org/zips/train2017.zip'
    IMG_ZIP     = f'{DRIVE_DATA_DIR}/train2017.zip'

ANN_ZIP_URL = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
ANN_ZIP     = f'{DRIVE_DATA_DIR}/annotations.zip'
ANN_DIR     = f'{LOCAL_DATA_DIR}/annotations'

# Download to Drive (if not there), then unzip to LOCAL disk
if not os.path.exists(ANN_ZIP):
    print('Downloading annotations zip to Drive...')
    !wget -q -O {ANN_ZIP} {ANN_ZIP_URL}
if not os.path.exists(ANN_DIR):
    print('Unzipping annotations to local disk...')
    !unzip -q {ANN_ZIP} -d {LOCAL_DATA_DIR}
else:
    print('Annotations already on local disk.')

if not os.path.exists(IMG_ZIP):
    print(f'Downloading images zip to Drive...')
    !wget -q -O {IMG_ZIP} {IMG_ZIP_URL}
if not os.path.exists(IMAGE_DIR):
    print('Unzipping images to local disk... (this is very fast)')
    !unzip -q {IMG_ZIP} -d {LOCAL_DATA_DIR}
else:
    print('Images already on local disk.')

print(f'Image dir:  {IMAGE_DIR}')
print(f'Ann file:   {ANN_FILE}')

Unzipping annotations to local disk...
Unzipping images to local disk... (this is very fast)
Image dir:  /content/dataset/train2017
Ann file:   /content/dataset/annotations/instances_train2017.json


## 4. Run teacher precomputation
Cache is saved to Google Drive and survives session disconnects.

In [ ]:
import os, sys, shutil
sys.path.insert(0, '/content/picosam3')

import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from PIL import Image
from pycocotools.coco import COCO
import wandb
import glob

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
from model_compression.utils import pad_bbox_to_square

CACHE_DIR        = '/content/teacher_sam3_logits'
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/picosam3/teacher_sam3_logits_backup'
IMAGE_SIZE       = 96
MIN_ROI_SIZE     = 4
VIS_INTERVAL     = 5000

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# Resume from the most recent backup tar.gz if one exists; otherwise rsync
# any loose cache files left over from an older session.
backups = glob.glob(os.path.join(DRIVE_BACKUP_DIR, '*.tar.gz'))
if backups:
    latest_backup = max(backups, key=os.path.getmtime)
    print(f"Extracting latest backup {latest_backup}...")
    os.system(f'tar -xzf {latest_backup} -C /content')
else:
    OLD_DRIVE_CACHE = '/content/drive/MyDrive/picosam3/teacher_sam3_logits'
    if os.path.exists(OLD_DRIVE_CACHE):
        print("Syncing loose cache from Drive (slow)...")
        os.system(f'rsync -a {OLD_DRIVE_CACHE}/ {CACHE_DIR}/')

print(f'Local Cache dir:  {CACHE_DIR}')
print(f'Drive Backup dir: {DRIVE_BACKUP_DIR}')
print(f'Image dir:  {IMAGE_DIR}')
print(f'Ann file:   {ANN_FILE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE"}')


Fast syncing: Extracting latest backup /content/drive/MyDrive/picosam3/teacher_sam3_logits_backup/cache_backup_752954.tar.gz to local disk...
Local Cache dir:  /content/teacher_sam3_logits
Drive Backup dir: /content/drive/MyDrive/picosam3/teacher_sam3_logits_backup
Image dir:  /content/dataset/train2017
Ann file:   /content/dataset/annotations/instances_train2017.json
GPU: NVIDIA L4


In [ ]:
def cache_path_for_ann(ann_id):
    return os.path.join(CACHE_DIR, f'ann_{ann_id}.pt')

def bbox_to_cxcywh_normalized(bbox, img_w, img_h):
    x, y, w, h = bbox
    return [(x + w/2)/img_w, (y + h/2)/img_h, w/img_w, h/img_h]

wandb.init(project='PicoSAM3-teacher-cache', name='sam3_colab', config={
    'image_size': IMAGE_SIZE, 'teacher': 'sam3', 'dataset': 'val2017' if USE_VAL else 'train2017'
})

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = build_sam3_image_model(
    device='cuda',
    eval_mode=True,
    load_from_HF=True,
    enable_segmentation=True,
)

processor = Sam3Processor(model=model, device=device, confidence_threshold=0.0)

coco = COCO(ANN_FILE)
anns = [a for a in coco.loadAnns(coco.getAnnIds())
        if 'segmentation' in a and a.get('iscrowd', 0) == 0]
anns_by_img = {}
for ann in anns:
    anns_by_img.setdefault(ann['image_id'], []).append(ann)

print(f'Total annotations: {len(anns)}')
print(f'Total images:      {len(anns_by_img)}')

processed = skipped_cached = skipped_small = skipped_empty = 0
BACKUP_INTERVAL = 25000

with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    for image_id, ann_list in tqdm(anns_by_img.items(), desc='Caching teacher masks'):
        to_process = [a for a in ann_list if not os.path.exists(cache_path_for_ann(a['id']))]
        skipped_cached += len(ann_list) - len(to_process)
        if not to_process:
            continue

        img_info = coco.loadImgs(image_id)[0]
        img_path = os.path.join(IMAGE_DIR, img_info.get('file_name', f'{image_id:012d}.jpg'))
        if not os.path.exists(img_path):
            continue

        image = Image.open(img_path).convert('RGB')
        img_np = np.array(image)
        img_h, img_w = img_np.shape[:2]
        state = processor.set_image(image)

        for ann in to_process:
            box_norm = bbox_to_cxcywh_normalized(ann['bbox'], img_w, img_h)
            processor.reset_all_prompts(state)
            state = processor.add_geometric_prompt(box=box_norm, label=True, state=state)

            if 'masks' not in state or len(state['masks']) == 0:
                skipped_empty += 1
                continue

            scores = state['scores'].to(torch.float32).cpu().numpy()
            best = scores.argmax()
            prob_full   = state['masks_logits'][best, 0].to(torch.float32).cpu().numpy()
            teacher_score = float(scores[best])

            rx1, ry1, rx2, ry2 = pad_bbox_to_square(ann['bbox'], img_w, img_h, padding=0.1)
            if (rx2-rx1) < MIN_ROI_SIZE or (ry2-ry1) < MIN_ROI_SIZE:
                skipped_small += 1
                continue

            roi_mask = prob_full[ry1:ry2, rx1:rx2]
            if roi_mask.size == 0:
                skipped_empty += 1
                continue

            roi_t = torch.tensor(roi_mask).unsqueeze(0).unsqueeze(0)
            roi_t = F.interpolate(roi_t, size=(IMAGE_SIZE, IMAGE_SIZE),
                                  mode='bilinear', align_corners=False).squeeze(0)
            eps = 1e-7
            roi_logits = torch.log(roi_t.clamp(eps, 1-eps) / (1 - roi_t.clamp(eps, 1-eps)))

            torch.save({
                'logits':   roi_logits.to(torch.float16).cpu(),
                'score':    teacher_score,
                'bbox':     ann['bbox'],
                'roi':      (rx1, ry1, rx2, ry2),
                'image_id': int(image_id),
            }, cache_path_for_ann(ann['id']))

            processed += 1
            if processed % 500 == 0:
                print(f'Processed {processed} | cached {skipped_cached} | small {skipped_small} | empty {skipped_empty}')
                wandb.log({'processed': processed, 'skipped_cached': skipped_cached})

            if processed % BACKUP_INTERVAL == 0:
                total_done = skipped_cached + processed
                print(f"\nBacking up cache to Google Drive (Total finished: {total_done})...")
                backup_file = f'{DRIVE_BACKUP_DIR}/cache_backup_{total_done}.tar.gz'
                os.system(f'tar -czf {backup_file} -C /content teacher_sam3_logits')
                print(f"Backup saved to {backup_file}")

print("\nCreating final backup...")
os.system(f'tar -czf {DRIVE_BACKUP_DIR}/cache_backup_final.tar.gz -C /content teacher_sam3_logits')

print(f'\nDone. Processed: {processed}, Skipped cached: {skipped_cached}, Small: {skipped_small}, Empty: {skipped_empty}')
wandb.finish()


config.json: 0.00B [00:00, ?B/s]

sam3.pt:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

loading annotations into memory...
Done (t=13.04s)
creating index...
index created!
Total annotations: 849949
Total images:      117266


Caching teacher masks:  85%|████████▍ | 99533/117266 [02:41<1:18:45,  3.75it/s]

Processed 500 | cached 752954 | small 473 | empty 0


...

Caching teacher masks: 100%|█████████▉| 116938/117266 [3:47:15<01:30,  3.61it/s]

Processed 96000 | cached 752954 | small 515 | empty 0


Caching teacher masks: 100%|██████████| 117266/117266 [3:48:52<00:00,  8.54it/s]



Creating final backup...

Done. Processed: 96480, Skipped cached: 752954, Small: 515, Empty: 0


processed,▁▁▁▂▂▂▂▂▂▂▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇███
skipped_cached,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
processed,96000
skipped_cached,752954


## 5. Verify cache
Counts how many `.pt` files were saved to Drive.

In [ ]:
cache_files = [f for f in os.listdir(CACHE_DIR) if f.endswith('.pt')]
print(f'Cache files in Drive: {len(cache_files)}')
if cache_files:
    sample = torch.load(os.path.join(CACHE_DIR, cache_files[0]), map_location='cpu')
    print(f'Sample keys:   {list(sample.keys())}')
    print(f'Logits shape:  {sample["logits"].shape}')
    print(f'Score:         {sample["score"]:.4f}')

Cache files in Drive: 850967
Sample keys:   ['logits', 'score', 'bbox', 'roi', 'image_id']
Logits shape:  torch.Size([1, 96, 96])
Score:         0.9531
